# DSC 670 Week 5 Exercise: Wikipedia RAG Chat

**Student:** Adjovi Gloria  
**Course:** DSC 670 – Advanced Uses of Generative AI  
**Topic:** Retrieval-Augmented Generation  
**Wikipedia source:** *Large language model*

## Project Purpose

This notebook builds a retrieval-augmented generation chatbot using the Wikipedia article **“Large language model.”** I selected this page because it is directly connected to the concepts studied in DSC 670, including model training, adaptation, hallucination, bias, computational cost, and responsible use.

The purpose is not only to make a chatbot answer factual questions. The stronger objective is to test whether a small RAG pipeline can retrieve evidence from a long article and support graduate-level synthesis. The questions require comparison, interpretation, and evaluation rather than simple name or date lookup.

The notebook asks at least two separate questions and displays both the generated answers and the retrieved source passages so that the quality of the response can be evaluated.

## Research Questions

### Question 1
**Using the article, compare pre-training, fine-tuning, and in-context learning as mechanisms for adapting large language model behavior. What practical trade-offs would an organization face when choosing among them for domain-specific work?**

This question requires the system to retrieve information from multiple sections and synthesize it into a comparison. A strong answer should distinguish model training from prompt-time adaptation and should discuss practical trade-offs such as cost, flexibility, data needs, and maintenance.

### Question 2
**Synthesize the article’s discussion of hallucination, bias, interpretability, and computational cost. Which of these problems could retrieval-augmented generation plausibly reduce, and which would still require governance, evaluation, or model-level controls?**

This question is intentionally more difficult. It requires the model to identify several limitations, separate them conceptually, and then make a careful inference about what RAG can and cannot solve. The answer should not claim that retrieval automatically removes bias, guarantees truth, or solves computational cost.

## Why I Chose a Transparent RAG Design

I used a small and inspectable pipeline instead of a large orchestration framework. The system uses:

- one public Wikipedia article;
- paragraph-aware chunking;
- TF-IDF retrieval;
- cosine similarity;
- an OpenAI model for answer generation;
- visible source passages for evaluation.

This design makes the reasoning process easier to audit. If the chatbot produces a weak answer, I can inspect whether the problem came from the source text, chunking, retrieval, or generation. That separation is important because a fluent response can still be poorly grounded.

In [ ]:
# Install packages only when they are missing.

import importlib.util
import subprocess
import sys

required_packages = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "sklearn": "scikit-learn",
    "openai": "openai"
}

missing = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

## 1. Collecting and Preserving the Source

A RAG system is only as reliable as the material it retrieves. For that reason, the first step is to collect the article and save a local text copy.

Saving the article locally improves reproducibility. Wikipedia pages can change over time, so keeping the text file makes it possible to preserve the exact source used for this experiment. The code removes navigation, citations, tables, and other page elements that do not contribute to the main narrative.

In [ ]:
from pathlib import Path
import re
import requests
from bs4 import BeautifulSoup

ARTICLE_URL = "https://en.wikipedia.org/wiki/Large_language_model"
TEXT_FILE = Path("large_language_model_wikipedia.txt")

headers = {
    "User-Agent": "DSC670-RAG-Class-Exercise/1.0"
}

response = requests.get(ARTICLE_URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Remove non-narrative elements.
for tag in soup.select(
    "table, sup, style, script, nav, figure, .mw-editsection, "
    ".navbox, .infobox, .sidebar, .metadata, .reflist"
):
    tag.decompose()

content = soup.select_one("#mw-content-text")
if content is None:
    raise RuntimeError("Wikipedia article content was not found.")

sections = []
current_heading = "Introduction"

for element in content.select("h2, h3, p"):
    if element.name in {"h2", "h3"}:
        heading = element.get_text(" ", strip=True)
        heading = re.sub(r"\s+", " ", heading).strip()
        if heading:
            current_heading = heading
    elif element.name == "p":
        paragraph = element.get_text(" ", strip=True)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if len(paragraph) >= 60:
            sections.append({
                "heading": current_heading,
                "text": paragraph
            })

article_text = "\n\n".join(
    f"[{item['heading']}]\n{item['text']}"
    for item in sections
)

TEXT_FILE.write_text(article_text, encoding="utf-8")

print("Article saved to:", TEXT_FILE.resolve())
print("Paragraphs retained:", len(sections))
print("Characters saved:", len(article_text))
print("\nPreview:\n")
print(article_text[:1200])

## Commentary on Source Quality

The cleaned preview should contain readable article paragraphs and section labels. I included section headings inside the saved text because they provide useful context during retrieval. A paragraph describing “fine-tuning,” for example, is more meaningful when the section title is preserved.

Wikipedia is acceptable for this classroom demonstration because the assignment specifically requires a Wikipedia page. However, I would not treat it as the only authority for a production system. For high-stakes use, I would retrieve from primary research papers, official technical documentation, governance policies, or internal organizational sources.

## 2. Chunking Strategy

Chunking determines how much context is available to the retriever. Very small chunks can separate definitions from supporting explanations. Very large chunks can lower retrieval precision because they contain too many unrelated ideas.

I used paragraph-aware chunks with moderate overlap. The design attempts to preserve complete ideas while still allowing the retriever to isolate relevant sections. The overlap reduces the risk that an important transition sentence appears at the edge of a chunk and is lost.

In [ ]:
def build_chunks(section_items, target_words=220, overlap_words=45):
    chunks = []
    buffer_words = []
    buffer_headings = []

    for item in section_items:
        paragraph_words = item["text"].split()
        heading = item["heading"]

        # Add heading context once for each paragraph.
        paragraph_block = [f"SECTION: {heading}"] + paragraph_words

        if len(buffer_words) + len(paragraph_block) <= target_words:
            buffer_words.extend(paragraph_block)
            buffer_headings.append(heading)
        else:
            if buffer_words:
                chunks.append({
                    "text": " ".join(buffer_words),
                    "headings": sorted(set(buffer_headings))
                })

            overlap = buffer_words[-overlap_words:] if buffer_words else []
            buffer_words = overlap + paragraph_block
            buffer_headings = [heading]

    if buffer_words:
        chunks.append({
            "text": " ".join(buffer_words),
            "headings": sorted(set(buffer_headings))
        })

    return chunks


chunks = build_chunks(sections)

print("Number of chunks:", len(chunks))
print("\nExample chunk headings:", chunks[0]["headings"])
print("\nExample chunk text:\n")
print(chunks[0]["text"][:1000])

## 3. Retrieval Method

The retriever uses TF-IDF and cosine similarity. TF-IDF emphasizes terms that are distinctive within a chunk, while cosine similarity measures how closely the question matches each chunk.

This method is appropriate for a focused experiment because it is transparent and inexpensive. I can inspect similarity scores and retrieved passages directly. Its main limitation is that it relies heavily on shared vocabulary. If a user asks a conceptually related question using very different wording, embedding-based retrieval may perform better.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunk_texts = [chunk["text"] for chunk in chunks]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1
)

chunk_matrix = vectorizer.fit_transform(chunk_texts)


def retrieve(question, top_k=6):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, chunk_matrix).flatten()
    top_indices = scores.argsort()[::-1][:top_k]

    results = []
    for index in top_indices:
        results.append({
            "chunk_id": int(index),
            "score": float(scores[index]),
            "headings": chunks[index]["headings"],
            "text": chunks[index]["text"]
        })

    return results

## Retrieval Inspection Before Generation

Before asking the language model to answer, I inspect the retrieved passages. This is important because the generator cannot produce a grounded answer if the retriever does not supply the necessary evidence.

For Question 1, I expect passages related to training, fine-tuning, prompting, or in-context learning. For Question 2, I expect passages related to hallucination, bias, limitations, evaluation, interpretability, or computing requirements.

In [ ]:
question_1 = (
    "Using the article, compare pre-training, fine-tuning, and in-context "
    "learning as mechanisms for adapting large language model behavior. "
    "What practical trade-offs would an organization face when choosing "
    "among them for domain-specific work?"
)

preview_results = retrieve(question_1, top_k=6)

for result in preview_results:
    print(
        f"Chunk {result['chunk_id']} | "
        f"Similarity: {result['score']:.3f} | "
        f"Sections: {', '.join(result['headings'])}"
    )
    print(result["text"][:700])
    print("-" * 100)

## 4. Connecting Retrieval to the OpenAI Model

The OpenAI model receives only the top retrieved passages, not the entire article. The prompt instructs the model to distinguish article-supported claims from analytical inference.

That distinction matters for the second question. The Wikipedia article may describe limitations of large language models, while the question asks which limitations RAG could reduce. The model is allowed to make a cautious inference, but it must label that reasoning and avoid presenting it as a direct quotation from the source.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    api_key = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=api_key)

MODEL_NAME = "gpt-4o-mini"

print("OpenAI client configured.")

## 5. RAG Chat Function

The function below performs four tasks:

1. retrieve relevant chunks;
2. assemble a source context;
3. generate a grounded answer;
4. display the evidence used.

The output includes similarity scores and article sections so I can evaluate whether the answer is supported. This is more useful than displaying only the final response because it exposes the retrieval process.

In [ ]:
def ask_rag_bot(question, top_k=6, show_sources=True):
    retrieved = retrieve(question, top_k=top_k)

    context_blocks = []
    for item in retrieved:
        section_names = ", ".join(item["headings"])
        context_blocks.append(
            f"[Chunk {item['chunk_id']} | Sections: {section_names}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    user_prompt = f"""
Use the retrieved Wikipedia context to answer the question.

Requirements:
- Ground factual claims in the provided context.
- Synthesize information from multiple chunks when needed.
- Clearly label any reasonable inference as an inference.
- Do not invent details.
- If the retrieved context is insufficient, say so directly.
- Use a professional graduate-level tone.
- Organize the answer with short headings or paragraphs when helpful.

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful retrieval-augmented generation assistant. "
                    "Use only the supplied context for factual claims and distinguish "
                    "source evidence from inference."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content.strip()

    print("QUESTION")
    print(question)
    print("\nANSWER")
    print(answer)

    if show_sources:
        print("\nRETRIEVED EVIDENCE")
        for item in retrieved:
            print(
                f"\nChunk {item['chunk_id']} | "
                f"Similarity: {item['score']:.3f} | "
                f"Sections: {', '.join(item['headings'])}"
            )
            print(item["text"])

    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved
    }

# Question 1: Model Adaptation Strategies

This question tests whether the RAG system can compare multiple mechanisms rather than retrieve one isolated fact. The expected answer should distinguish:

- broad learning during pre-training;
- targeted parameter updates during fine-tuning;
- temporary behavior adaptation through examples or instructions in the prompt.

The organizational trade-off should address issues such as cost, infrastructure, data requirements, speed of updating, flexibility, and control.

In [ ]:
result_1 = ask_rag_bot(question_1, top_k=6)

## Evaluation of Question 1

After running the notebook, I will evaluate the answer using the following criteria:

1. **Conceptual distinction:** Does the response clearly separate pre-training, fine-tuning, and in-context learning?
2. **Evidence coverage:** Do the retrieved passages contain support for each mechanism?
3. **Synthesis quality:** Does the answer compare the methods instead of presenting disconnected definitions?
4. **Practical relevance:** Does it connect the technical differences to organizational decisions?
5. **Grounding:** Are the major factual claims visible in the retrieved evidence?

A strong response should explain that pre-training is the broadest and most resource-intensive stage, fine-tuning changes model behavior using targeted data, and in-context learning adapts behavior at inference time without updating model weights. The trade-off analysis should recognize that lower-cost and faster approaches may offer less persistent control than model-level adaptation.

# Question 2: Limitations, RAG, and Governance

The second question is more demanding because it combines source retrieval with analytical judgment. The bot must first identify limitations discussed in the article and then evaluate which limitations RAG might reduce.

A strong response should avoid overstating RAG. Retrieval may improve access to external or current evidence and may reduce some unsupported factual generation, but it does not automatically eliminate bias, solve interpretability, guarantee source quality, or remove the need for governance.

In [ ]:
question_2 = (
    "Synthesize the article's discussion of hallucination, bias, "
    "interpretability, and computational cost. Which of these problems "
    "could retrieval-augmented generation plausibly reduce, and which "
    "would still require governance, evaluation, or model-level controls?"
)

result_2 = ask_rag_bot(question_2, top_k=7)

## Evaluation of Question 2

I will evaluate this response using a stricter standard because it contains both retrieval and inference.

1. **Coverage:** Does the response address hallucination, bias, interpretability, and computational cost?
2. **Separation of issues:** Does it avoid treating all limitations as the same problem?
3. **RAG boundary:** Does it explain that retrieval can support factual grounding without claiming that it solves every model risk?
4. **Governance awareness:** Does it recognize the continuing need for source review, bias testing, monitoring, human oversight, and model evaluation?
5. **Transparency:** Are inference-based claims clearly distinguished from article-supported claims?

The most important result is whether the chatbot explains the boundary of RAG. A mature answer should state that better retrieval can reduce some knowledge gaps or unsupported answers, while biased sources, biased model behavior, opaque reasoning, and high training or inference cost remain broader system-level concerns.

## Optional Robustness Test

A RAG system should not answer confidently when the selected source does not contain the necessary information. This optional question asks for a specific local policy recommendation that cannot be supported by a general Wikipedia article about large language models.

The expected behavior is to state that the article is insufficient rather than invent a recommendation.

In [ ]:
result_3 = ask_rag_bot(
    "Based only on this article, what exact AI procurement policy should the City of Omaha adopt in 2027?",
    top_k=5
)

## Overall Commentary on the Results

This notebook should be evaluated as a complete information pipeline rather than only as a language-model demonstration. The final answer depends on source quality, chunking, retrieval, prompt design, and model generation.

The first question tests comparative synthesis across model adaptation methods. The second tests whether the system can retrieve multiple limitations and reason carefully about the boundary between RAG and broader AI governance. The optional third question tests whether the bot recognizes when the source is insufficient.

The visible evidence is essential. If the answer is weak but the correct passages were retrieved, the generation prompt may need improvement. If the passages are irrelevant, the retrieval method or chunking strategy is the likely problem. This separation makes the system easier to diagnose and improve.

## Limitations of This Implementation

The retriever uses TF-IDF, which depends on lexical overlap. It may miss relevant passages when the question and article use different terminology. An embedding-based retriever would likely perform better on conceptual similarity.

The source is also limited to one Wikipedia article. A real RAG system would usually use multiple curated documents, metadata filters, source ranking, and stronger citation controls. It would also require systematic evaluation rather than relying on only two questions.

Finally, RAG does not guarantee factual accuracy. It can retrieve incorrect, outdated, or biased material. The model can also misinterpret correct context. Responsible deployment therefore requires source governance, retrieval evaluation, answer evaluation, logging, and human oversight.

## Improvements for a Production-Quality Version

A stronger version of this system would include:

- embedding-based retrieval;
- multiple authoritative sources;
- metadata and section filters;
- reranking of retrieved passages;
- sentence-level citations;
- automated retrieval evaluation;
- answer faithfulness scoring;
- prompt-injection defenses;
- source freshness monitoring;
- access controls for private documents;
- human review for high-impact decisions.

These improvements show that RAG is not simply “adding documents to a prompt.” It is a complete system design problem involving information retrieval, model behavior, data governance, evaluation, and user trust.

## Ethical Considerations

This experiment uses a public Wikipedia article and does not process confidential or personal data. The OpenAI API key is entered through a hidden prompt and should not be saved inside the submitted notebook.

The main ethical risks are misinformation, overconfidence, source bias, and false authority. Displaying retrieved evidence improves transparency, but users may still assume that a fluent answer is correct. The system therefore instructs the model to acknowledge insufficient context and distinguish factual support from inference.

For organizational use, ethical safeguards would also need to address source permissions, privacy, retention, access control, bias testing, monitoring, and accountability for decisions influenced by the chatbot.

## Conclusion

This project demonstrates a small but complete retrieval-augmented generation workflow using a Wikipedia article related to generative AI. The chatbot retrieves relevant passages, answers two graduate-level questions, and displays the evidence used.

The main lesson is that RAG quality depends on more than the language model. The source, chunking strategy, retrieval method, prompt, and evaluation process all influence the final result. RAG can improve grounding and access to external knowledge, but it does not remove the need for governance, source validation, bias review, or human judgment.

## References

Wikipedia contributors. (n.d.). *Large language model*. Wikipedia.

OpenAI. (n.d.). *OpenAI API documentation*.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830.